# Reruns July studies after fixing the CNT [0,1]-vs-[-1,1] input-range bug

Produces `corrected_rerun/{correspondence_corrected_n264.csv, consistency_corrected_n80.csv, parametric_*_n200.csv, lpips_*, selfsup_*, subdivision_*, realism_*, inception_with_sym_n500.json}`. Feeds the order study's field-linear swap distance (.256) and the (unused) correspondence-regime / OT-sym recall macros.

**Needs:** DTD, the CNT checkpoint (post-fix).

**Maintained runnable path:** none. No `experiments/*.py` script covers this notebook's studies; run it as-is to reproduce or inspect its claims.

In [ ]:
!pip install -q pot
import os, sys, torch, torch.nn.functional as F, numpy as np, glob, random, inspect, time
from pathlib import Path
from PIL import Image
import pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DTD_ROOT = f'{BASE}/dtd/images'
OUT = f'{BASE}/corrected_rerun'; os.makedirs(OUT, exist_ok=True)
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

# what does the transform actually do?
from src.inference import core as cnt_core
try:
    print(inspect.getsource(cnt_core.build_content_transform))
except Exception as e:
    import src.inference.core as c
    print("could not fetch:", e)

# ---- THE loader. Always go through load_image, via a temp file if needed. ----
_tmp_counter = [0]
def cnt_load(x, size=IMG_SIZE):
    """x: path or PIL. Always routed through CNT's load_image."""
    if isinstance(x, (str, Path)):
        return load_image(Path(x), size).to(device)
    _tmp_counter[0] += 1
    fp = f'{TMP}/t{_tmp_counter[0] % 64}.png'
    x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)

def to_pil(dec):
    a = ((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def contrast(im): return float(np.asarray(im.convert('L')).std())

def get_gen_inputs(img):
    with torch.no_grad():
        blob = model.encode(img); layout = model.splat(blob)
        return blob, layout["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     layout["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf
        layout["layer_level_layouts"][-1][0]["score_img"] = si
        return model.decode(layout)
def build_style_maps(bf, si, fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        layout=model.splat(blob_ref); si=layout["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        feat_in=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=feat_in.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    plan=u*K*v.T; plan=plan/(plan.sum(1,keepdim=True)+1e-10); return plan@tgt

def build_d(ia, ib):
    ba,bfa,sia = get_gen_inputs(ia); bb,bfb,sib = get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sm_a[L][0],sm_b[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    interp=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_ot_sym(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sm_a[L][0],sm_b[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bfm=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        t_ab=sinkhorn_transport(af[ia],bfm[ib]); t_ba=sinkhorn_transport(bfm[ib],af[ia])
    nn_a=torch.cdist(af,af[ia]).argmin(1); nn_b=torch.cdist(bfm,bfm[ib]).argmin(1)
    interp=(0.5*(((1-eta)*af+eta*t_ab[nn_a])+(eta*bfm+(1-eta)*t_ba[nn_b]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_lin_pixel(d, eta):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sm_a[s]+eta*sm_b[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])

FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}
METHODS = list(FNS)
random.seed(1)
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print("\nsetup done on", device)

In [ ]:
N = 40
rows=[]
for i in range(N):
    p = dtd_path(random.choice(cats), 11000+i)
    src = Image.open(p).convert('RGB').resize((IMG_SIZE,)*2)
    t = cnt_load(p)
    with torch.no_grad():
        rt_model = to_pil(model.decode(model.splat(model.encode(t))))
    blob, bf, si = get_gen_inputs(t)
    rt_sm = to_pil(decode_with_style_maps(blob, build_style_maps(bf, si, FMS)))
    rows.append(dict(c_src=contrast(src), c_model=contrast(rt_model), c_sm=contrast(rt_sm)))
    if i%10==0: print(f"  {i}/{N}")
dd = pd.DataFrame(rows)
print(f"\n=== ROUND-TRIP FIDELITY, correct loading (n={N}) ===")
print(f"  source contrast            {dd.c_src.median():6.1f}")
print(f"  model.decode(splat(enc))   {dd.c_model.median():6.1f}   retained {100*(dd.c_model/dd.c_src).median():.0f}%")
print(f"  our style-map decode       {dd.c_sm.median():6.1f}   retained {100*(dd.c_sm/dd.c_src).median():.0f}%")
print(f"\n  previously (wrong [0,1] input): ~24% retained")

In [ ]:
from torchvision.models import inception_v3, Inception_V3_Weights
import timm
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in list(_inc.parameters())+list(dino.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)

@torch.no_grad()
def _feats(net, size, pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255.
            for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device)
        b=F.interpolate(b,size=size,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(net(b).cpu().numpy())
    return np.concatenate(out)
inc_feats  = lambda p: _feats(_inc, 299, p)
dino_feats = lambda p: _feats(dino, 518, p)

N_PAIRS = 100
rng = random.Random(4242)
PAIRS=[]
while len(PAIRS)<N_PAIRS:
    a,b = rng.sample(cats,2)
    PAIRS.append((dtd_path(a,20000+len(PAIRS)), dtd_path(b,40000+len(PAIRS))))

# generate all midpoints with CORRECT loading
mids = {m: [] for m in METHODS}
srcs = []
t0=time.time()
for k,(pa,pb) in enumerate(PAIRS):
    ta, tb = cnt_load(pa), cnt_load(pb)
    d = build_d(ta, tb)
    for m,fn in FNS.items(): mids[m].append(to_pil(fn(d,0.5)))
    srcs.append((to_pil(ta*1.0), to_pil(tb*1.0)))     # sources as the model saw them
    if k%25==0: print(f"  {k}/{N_PAIRS} ({(time.time()-t0)/60:.1f}m)")

def build_manifold(featfn, n=400, K=3):
    imgs=[Image.open(dtd_path(random.choice(cats),50000+i)).convert('RGB') for i in range(n)]
    Phi=featfn(imgs); Phi=Phi[~np.isnan(Phi).any(axis=1)]
    D=np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
    rad=np.sort(D,axis=1)[:,K-1]; keep=rad<=np.median(rad)
    return Phi[keep], rad[keep]

res={}
for oname, featfn in [('inception', inc_feats), ('dinov2', dino_feats)]:
    Phi_r, radii = build_manifold(featfn)
    def realism(phi):
        dd=np.linalg.norm(Phi_r-phi,axis=1)+1e-9; return float(np.max(radii/dd))
    res[oname] = {m: np.array([realism(f) for f in featfn(mids[m])]) for m in METHODS}

c_src = np.median([0.5*(contrast(a)+contrast(b)) for a,b in srcs])
print(f"\n=== REALISM, CORRECTED PIPELINE (n={N_PAIRS}) ===")
print(f"  source contrast {c_src:.1f}")
for oname in ['inception','dinov2']:
    sc=res[oname]
    print(f"\n--- {oname} ---")
    print(f"  {'method':16} {'mean R':>8} {'%R>=1':>7} {'contrast':>9}")
    for m in METHODS:
        print(f"  {m:16} {sc[m].mean():8.3f} {100*(sc[m]>=1).mean():6.0f}% "
              f"{np.median([contrast(i) for i in mids[m]]):9.1f}")
    print(f"  --- ot_sym vs ---")
    for base in ['ot','pixel_linear','texton_linear']:
        a,b = sc['ot_sym'], sc[base]
        w=int((a>b).sum()); p=wilcoxon(a,b,alternative='greater').pvalue
        print(f"    {base:14}: {w}/{N_PAIRS}  p={p:.2e}  {'WINS' if w>N_PAIRS/2 else 'loses'}")

np.save(f'{OUT}/realism_corrected_n{N_PAIRS}.npy', res, allow_pickle=True)

# qualitative check
fig, axes = plt.subplots(4, 6, figsize=(17, 11.5))
for r,k in enumerate([0, 7, 19, 33]):
    A,B = srcs[k]
    panels = [('A',A)] + [(m, mids[m][k]) for m in METHODS] + [('B',B)]
    for c,(lbl,im) in enumerate(panels):
        axes[r,c].imshow(im); axes[r,c].axis('off')
        axes[r,c].set_title(f"{lbl}  {contrast(im):.0f}", fontsize=9)
plt.suptitle("Corrected pipeline — contrast under each panel", fontsize=13)
plt.tight_layout(); plt.savefig(f'{OUT}/qualitative_corrected.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
TARGET_MEAN, TARGET_RMS = 0.5, 0.18

def gen_oriented_noise(f0, theta_deg, size=IMG_SIZE, f_bw=0.35, th_bw_deg=18.0, seed=0):
    """Stationary bandpass noise with known radial frequency and orientation."""
    rng = np.random.default_rng(seed)
    n = rng.standard_normal((size, size))
    Fn = np.fft.fftshift(np.fft.fft2(n))
    cy = cx = size // 2
    yy, xx = np.mgrid[0:size, 0:size]
    fy, fx = (yy-cy).astype(float), (xx-cx).astype(float)
    r = np.sqrt(fx**2 + fy**2) + 1e-9
    ang = np.arctan2(fy, fx)
    radial = np.exp(-((np.log(r/f0))**2)/(2*f_bw**2))
    th0 = np.deg2rad(theta_deg)
    dth = np.angle(np.exp(1j*2*(ang-th0)))/2.0
    angular = np.exp(-(dth**2)/(2*np.deg2rad(th_bw_deg)**2))
    filt = radial*angular; filt[cy,cx] = 0.0
    img = np.real(np.fft.ifft2(np.fft.ifftshift(Fn*filt)))
    img = (img-img.mean())/(img.std()+1e-9)
    img = np.clip(img*TARGET_RMS + TARGET_MEAN, 0, 1)
    return np.repeat(img[:,:,None], 3, axis=2).astype(np.float32)

def arr_to_pil(a):
    return Image.fromarray((np.clip(a,0,1)*255).astype(np.uint8))

def estimate_params(rgb_or_pil):
    a = np.asarray(rgb_or_pil.convert('L') if isinstance(rgb_or_pil, Image.Image)
                   else (rgb_or_pil.mean(axis=2)*255)).astype(float)
    g = a - a.mean()
    P = np.abs(np.fft.fftshift(np.fft.fft2(g)))**2
    H,W = P.shape; cy,cx = H//2, W//2
    yy,xx = np.mgrid[0:H,0:W]
    fy,fx = (yy-cy).astype(float), (xx-cx).astype(float)
    r = np.sqrt(fx**2+fy**2)
    m = (r>2)&(r<min(H,W)/2-1); w = P[m]
    if w.sum()<=0: return np.nan, np.nan
    f_est = float((r[m]*w).sum()/w.sum())
    aa = np.arctan2(fy[m], fx[m])
    c = (w*np.cos(2*aa)).sum(); s = (w*np.sin(2*aa)).sum()
    return f_est, float(np.degrees(0.5*np.arctan2(s,c)) % 180.0)

def ang_err(a,b):
    d = abs(a-b) % 180.0; return min(d, 180.0-d)

def spectral_bimodality(pil, fa_e, fb_e, w=0.18):
    g = np.asarray(pil.convert('L')).astype(float); g = g-g.mean()
    P = np.abs(np.fft.fftshift(np.fft.fft2(g)))**2
    H,W = P.shape; cy,cx = H//2, W//2
    yy,xx = np.mgrid[0:H,0:W]
    r = np.sqrt((xx-cx)**2+(yy-cy)**2)
    band = lambda f: float(P[(r>f*(1-w))&(r<f*(1+w))].sum())
    return (band(fa_e)+band(fb_e))/(band(np.sqrt(fa_e*fb_e))+1e-9)

# --- gate: does CNT round-trip these correctly NOW? ---
print("=== round-trip gate, corrected loading ===")
for f0, th in [(10,0),(16,45),(24,90),(34,135)]:
    im = arr_to_pil(gen_oriented_noise(f0, th, seed=int(f0*7+th)))
    t = cnt_load(im)
    rt = to_pil(model.decode(model.splat(model.encode(t))))
    fi,ti = estimate_params(im); fr,tr = estimate_params(rt)
    print(f"  f={f0:2d} θ={th:3d}: freq {fi:5.1f}→{fr:5.1f} ({abs(fr-fi)/fi:.3f})  "
          f"θ {ti:5.1f}→{tr:5.1f} ({ang_err(tr,ti):.1f}°)  "
          f"contrast {contrast(im):.1f}→{contrast(rt):.1f}")

In [ ]:
N_PAIRS = 200
F_LO, F_HI = 8.0, 40.0
TH_JITTER = 6.0
ALLM = METHODS + ['crossfade']
rng = np.random.default_rng(0)
records=[]
t0=time.time()
for i in range(N_PAIRS):
    fa, fb = np.exp(rng.uniform(np.log(F_LO), np.log(F_HI), size=2))
    if fb<fa: fa,fb = fb,fa
    if fb/fa < 1.6:
        fb = fa*1.6
        if fb>F_HI: fa,fb = F_HI/1.6, F_HI
    f_mid = np.sqrt(fa*fb)
    th = rng.uniform(0,180)
    sa,sb,sm_ = (int(rng.integers(1e6)) for _ in range(3))
    imA = arr_to_pil(gen_oriented_noise(fa, th+rng.uniform(-TH_JITTER,TH_JITTER), seed=sa))
    imB = arr_to_pil(gen_oriented_noise(fb, th+rng.uniform(-TH_JITTER,TH_JITTER), seed=sb))
    imGT= arr_to_pil(gen_oriented_noise(f_mid, th, seed=sm_))

    f_gt, th_gt = estimate_params(imGT)
    fa_e,_ = estimate_params(imA); fb_e,_ = estimate_params(imB)
    c_gt = contrast(imGT)

    d = build_d(cnt_load(imA), cnt_load(imB))
    outs = {m: to_pil(fn(d,0.5)) for m,fn in FNS.items()}
    outs['crossfade'] = arr_to_pil(0.5*np.asarray(imA).astype(float)/255
                                   + 0.5*np.asarray(imB).astype(float)/255)

    rec = dict(pair=i, fa=fa, fb=fb, fa_est=fa_e, fb_est=fb_e, f_gt_est=f_gt,
               bimod_gt=spectral_bimodality(imGT, fa_e, fb_e), c_gt=c_gt)
    for m,img in outs.items():
        fe, te = estimate_params(img)
        rec[f'{m}_f'] = fe
        rec[f'{m}_f_relerr'] = abs(fe-f_gt)/f_gt
        rec[f'{m}_th_err'] = ang_err(te, th_gt)
        rec[f'{m}_bimod'] = spectral_bimodality(img, fa_e, fb_e)
        rec[f'{m}_contrast'] = contrast(img)
    records.append(rec)
    if i%25==0: print(f"  {i}/{N_PAIRS} ({(time.time()-t0)/60:.1f}m)")

df = pd.DataFrame(records); df.to_csv(f'{OUT}/parametric_freq_corrected_n{N_PAIRS}.csv', index=False)

fa_est, fb_est = df.fa_est.values, df.fb_est.values
t_imp = {}
for m in ALLM + ['gt']:
    col = df.f_gt_est if m=='gt' else df[f'{m}_f']
    t_imp[m] = (np.log(col.values)-np.log(fa_est))/(np.log(fb_est)-np.log(fa_est))

print(f"\n=== FREQUENCY, corrected pipeline (n={N_PAIRS}) ===")
print(f"  {'method':16} {'|Δf|/f ↓':>10} {'|t-0.5| ↓':>11} {'bimod':>8} {'contrast':>10}")
print(f"  {'ground truth':16} {'-':>10} {np.median(abs(t_imp['gt']-0.5)):11.3f} "
      f"{df.bimod_gt.median():8.2f} {df.c_gt.median():10.1f}")
for m in ALLM:
    print(f"  {m:16} {df[f'{m}_f_relerr'].median():10.3f} {np.median(abs(t_imp[m]-0.5)):11.3f} "
          f"{df[f'{m}_bimod'].median():8.2f} {df[f'{m}_contrast'].median():10.1f}")

print(f"\n=== paired comparisons ===")
for tgt in ['ot','ot_sym']:
    print(f"  --- {tgt} vs ---")
    for base in [m for m in ALLM if m!=tgt]:
        a,b = df[f'{tgt}_f_relerr'], df[f'{base}_f_relerr']
        w=int((a<b).sum()); p=wilcoxon(a,b,alternative='less').pvalue
        ap,bp = np.abs(t_imp[tgt]-0.5), np.abs(t_imp[base]-0.5)
        w2=int((ap<bp).sum()); p2=wilcoxon(ap,bp,alternative='less').pvalue
        print(f"    {base:14} recovery {w:3d}/{N_PAIRS} p={p:.1e} | "
              f"placement {w2:3d}/{N_PAIRS} p={p2:.1e}")

In [ ]:
print(f"  {'f0':>5} {'θ':>5} {'|Δf|/f':>9} {'Δθ':>7} {'contrast retained':>18}")
for f0 in [8,12,16,20,24,28,34,40]:
    for th in [0,45,90]:
        im = arr_to_pil(gen_oriented_noise(f0, th, seed=int(f0*13+th)))
        rt = to_pil(model.decode(model.splat(model.encode(cnt_load(im)))))
        fi,ti = estimate_params(im); fr,tr = estimate_params(rt)
        print(f"  {f0:5d} {th:5d} {abs(fr-fi)/fi:9.3f} {ang_err(tr,ti):7.1f} "
              f"{100*contrast(rt)/contrast(im):17.0f}%")

In [ ]:
N_PAIRS = 200
F_FIXED = 20.0
DTH_MIN, DTH_MAX = 25.0, 75.0
ALLM = METHODS + ['crossfade']

def circ_mid(a,b):
    d = ((b-a+90.0) % 180.0) - 90.0
    return (a + d/2.0) % 180.0
def signed_ang(a,b):
    return ((a-b+90.0) % 180.0) - 90.0

def orientation_bimodality(pil, th_a_e, th_b_e, w_deg=12.0):
    g = np.asarray(pil.convert('L')).astype(float); g = g-g.mean()
    P = np.abs(np.fft.fftshift(np.fft.fft2(g)))**2
    H,W = P.shape; cy,cx = H//2, W//2
    yy,xx = np.mgrid[0:H,0:W]
    fy,fx = (yy-cy).astype(float), (xx-cx).astype(float)
    r = np.sqrt(fx**2+fy**2); ang = np.degrees(np.arctan2(fy,fx)) % 180.0
    ring = (r > F_FIXED*0.6) & (r < F_FIXED*1.5)
    def band(th):
        dd = np.abs(((ang-th+90) % 180) - 90)
        return float(P[ring & (dd < w_deg)].sum())
    return (band(th_a_e)+band(th_b_e))/(band(circ_mid(th_a_e,th_b_e))+1e-9)

rng = np.random.default_rng(1)
records=[]; t0=time.time()
for i in range(N_PAIRS):
    th_a = rng.uniform(0,180)
    dth  = rng.uniform(DTH_MIN,DTH_MAX)*rng.choice([-1,1])
    th_b = (th_a+dth) % 180.0
    th_gt = circ_mid(th_a, th_b)
    sa,sb,sm_ = (int(rng.integers(1e6)) for _ in range(3))
    imA  = arr_to_pil(gen_oriented_noise(F_FIXED, th_a,  seed=sa))
    imB  = arr_to_pil(gen_oriented_noise(F_FIXED, th_b,  seed=sb))
    imGT = arr_to_pil(gen_oriented_noise(F_FIXED, th_gt, seed=sm_))

    f_gt_e, th_gt_e = estimate_params(imGT)
    _, th_a_e = estimate_params(imA)
    _, th_b_e = estimate_params(imB)

    d = build_d(cnt_load(imA), cnt_load(imB))
    outs = {m: to_pil(fn(d,0.5)) for m,fn in FNS.items()}
    outs['crossfade'] = arr_to_pil(0.5*np.asarray(imA).astype(float)/255
                                   + 0.5*np.asarray(imB).astype(float)/255)

    span = signed_ang(th_b_e, th_a_e)
    rec = dict(pair=i, dth=abs(dth), th_a_est=th_a_e, th_b_est=th_b_e,
               th_gt_est=th_gt_e, f_gt_est=f_gt_e,
               bimod_gt=orientation_bimodality(imGT, th_a_e, th_b_e),
               c_gt=contrast(imGT))
    for m,img in outs.items():
        fe, te = estimate_params(img)
        rec[f'{m}_th_err']  = ang_err(te, th_gt_e)
        rec[f'{m}_f_drift'] = abs(fe-f_gt_e)/f_gt_e        # should stay ~0
        rec[f'{m}_bimod']   = orientation_bimodality(img, th_a_e, th_b_e)
        rec[f'{m}_contrast']= contrast(img)
        rec[f'{m}_t_imp']   = (signed_ang(te, th_a_e)/span) if abs(span)>1e-6 else np.nan
    records.append(rec)
    if i%25==0: print(f"  {i}/{N_PAIRS} ({(time.time()-t0)/60:.1f}m)")

dfo = pd.DataFrame(records); dfo.to_csv(f'{OUT}/parametric_orient_corrected_n{N_PAIRS}.csv', index=False)

print(f"\n=== ORIENTATION, corrected pipeline (n={N_PAIRS}), f fixed at {F_FIXED} ===")
print(f"  {'method':16} {'Δθ° ↓':>8} {'|t-0.5| ↓':>11} {'bimod':>8} {'f drift ↓':>10} {'contrast':>10}")
print(f"  {'ground truth':16} {'-':>8} {'-':>11} {dfo.bimod_gt.median():8.2f} {'-':>10} {dfo.c_gt.median():10.1f}")
for m in ALLM:
    dev = np.abs(dfo[f'{m}_t_imp']-0.5)
    print(f"  {m:16} {dfo[f'{m}_th_err'].median():8.2f} {np.nanmedian(dev):11.3f} "
          f"{dfo[f'{m}_bimod'].median():8.2f} {dfo[f'{m}_f_drift'].median():10.3f} "
          f"{dfo[f'{m}_contrast'].median():10.1f}")

print(f"\n=== paired comparisons ===")
for tgt in ['ot','ot_sym']:
    print(f"  --- {tgt} vs ---")
    for base in [m for m in ALLM if m!=tgt]:
        a,b = dfo[f'{tgt}_th_err'], dfo[f'{base}_th_err']
        ok = a.notna()&b.notna()
        w=int((a[ok]<b[ok]).sum()); p=wilcoxon(a[ok],b[ok],alternative='less').pvalue
        ap = np.abs(dfo[f'{tgt}_t_imp']-0.5); bp = np.abs(dfo[f'{base}_t_imp']-0.5)
        ok2 = ap.notna()&bp.notna()
        w2=int((ap[ok2]<bp[ok2]).sum()); p2=wilcoxon(ap[ok2],bp[ok2],alternative='less').pvalue
        print(f"    {base:14} orient {w:3d}/{ok.sum()} p={p:.1e} | "
              f"placement {w2:3d}/{ok2.sum()} p={p2:.1e}")

print(f"\n=== by endpoint gap (median Δθ error) ===")
print(f"  {'Δθ range':12} {'n':>4} " + "".join(f"{m[:11]:>13}" for m in ALLM))
for lo,hi in [(25,40),(40,60),(60,75)]:
    s = dfo[(dfo.dth>=lo)&(dfo.dth<hi)]
    if len(s)<5: continue
    print(f"  {lo}-{hi}°       {len(s):4d} " + "".join(f"{s[f'{m}_th_err'].median():13.2f}" for m in ALLM))

In [ ]:
!pip install -q opencv-python-headless
import cv2
sift = cv2.SIFT_create(nfeatures=2000)
bf_matcher = cv2.BFMatcher()

def corr_matches(pilA, pilB):
    a = np.asarray(pilA.convert('L').resize((IMG_SIZE,)*2))
    b = np.asarray(pilB.convert('L').resize((IMG_SIZE,)*2))
    ka,da = sift.detectAndCompute(a,None); kb,db = sift.detectAndCompute(b,None)
    if da is None or db is None or len(ka)<8 or len(kb)<8: return 0.0, 0
    good = [m for m,n in bf_matcher.knnMatch(da,db,k=2) if m.distance < 0.75*n.distance]
    if len(good)<8: return 0.0, len(good)
    src = np.float32([ka[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst = np.float32([kb[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    _,mask = cv2.findHomography(src,dst,cv2.RANSAC,5.0)
    return (float(mask.sum())/len(good) if mask is not None else 0.0), len(good)

def warp_level(pil, lvl, seed):
    r = random.Random(seed); w,h = pil.size
    if lvl==0: return pil
    shift=[0,4,10,20,36][min(lvl,4)]; ang=[0,0,3,8,15][min(lvl,4)]; sc=[1,1,1.05,1.12,1.25][min(lvl,4)]
    out = pil.transform(pil.size, Image.AFFINE,
                        (1,0,r.randint(-shift,shift),0,1,r.randint(-shift,shift)), Image.BILINEAR)
    if ang: out = out.rotate(r.uniform(-ang,ang), resample=Image.BILINEAR)
    if sc>1:
        cw,ch = int(w/sc), int(h/sc); l,t = (w-cw)//2,(h-ch)//2
        out = out.crop((l,t,l+cw,t+ch)).resize((w,h), Image.BILINEAR)
    return out

def overlap_crops(path, frac, seed):
    pil = Image.open(path).convert('RGB'); W,H = pil.size
    if min(W,H) < IMG_SIZE+8: return None
    r = random.Random(seed); off = int(IMG_SIZE*(1-frac))
    x0 = r.randint(0, max(0, W-IMG_SIZE-off-1)); y0 = r.randint(0, max(0, H-IMG_SIZE-1))
    a = pil.crop((x0,y0,x0+IMG_SIZE,y0+IMG_SIZE))
    b = pil.crop((x0+off,y0,x0+off+IMG_SIZE,y0+IMG_SIZE))
    return (a,b) if b.size==(IMG_SIZE,IMG_SIZE) else None

random.seed(7)
plist=[]
for lvl in range(5):
    for i in range(24):
        plist.append(dict(kind=f'warp{lvl}', pa=dtd_path(random.choice(cats),1000+lvl*100+i),
                          lvl=lvl, seed=1000+lvl*100+i))
for frac in [0.75,0.5,0.25]:
    got=0; tries=0
    while got<24 and tries<200:
        p = dtd_path(random.choice(cats), 3000+tries)
        if overlap_crops(p, frac, 3000+tries) is not None:
            plist.append(dict(kind=f'ovl{frac}', pa=p, frac=frac, seed=3000+tries)); got+=1
        tries+=1
for i in range(24):
    c = random.choice(cats); ps = sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg'))
    if len(ps)>=2:
        q = random.Random(5000+i).sample(ps,2)
        plist.append(dict(kind='same-cat', pa=q[0], pb=q[1], seed=5000+i))
for i in range(48):
    a,b = random.sample(cats,2)
    plist.append(dict(kind='cross', pa=dtd_path(a,6000+i), pb=dtd_path(b,9000+i), seed=6000+i))
random.shuffle(plist)
print(f"{len(plist)} pairs")

def realise(rec):
    if rec['kind'].startswith('warp'):
        A = Image.open(rec['pa']).convert('RGB').resize((IMG_SIZE,)*2)
        return A, warp_level(A, rec['lvl'], rec['seed'])
    if rec['kind'].startswith('ovl'):
        return overlap_crops(rec['pa'], rec['frac'], rec['seed'])
    A = Image.open(rec['pa']).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(rec['pb']).convert('RGB').resize((IMG_SIZE,)*2)
    return A,B

Phi_r, radii = build_manifold(dino_feats)
def realism_d(phi):
    dd=np.linalg.norm(Phi_r-phi,axis=1)+1e-9; return float(np.max(radii/dd))
SAME_IMG = tuple(f'warp{i}' for i in range(5)) + ('ovl0.75','ovl0.5','ovl0.25')

rows=[]; t0=time.time()
for k,rec in enumerate(plist):
    out = realise(rec)
    if out is None: continue
    A,B = out
    ir, nm = corr_matches(A,B)
    d = build_d(cnt_load(A), cnt_load(B))
    mids = {m: to_pil(fn(d,0.5)) for m,fn in FNS.items()}
    R = {m: realism_d(f) for m,f in zip(METHODS, dino_feats([mids[m] for m in METHODS]))}
    fA,fB = dino_feats([A,B])
    r = dict(pair=k, kind=rec['kind'],
             regime='same-image' if rec['kind'] in SAME_IMG else 'different-image',
             inlier=ir, n_match=nm, endpoint_dist=float(np.linalg.norm(fA-fB)))
    r.update({f'R_{m}': R[m] for m in METHODS})
    r['adv_sym_pixel'] = R['ot_sym']-R['pixel_linear']
    r['adv_sym_tex']   = R['ot_sym']-R['texton_linear']
    r['adv_ot_pixel']  = R['ot']-R['pixel_linear']
    rows.append(r)
    if k%30==0: print(f"  {k}/{len(plist)} ({(time.time()-t0)/60:.1f}m)")

dfc = pd.DataFrame(rows); dfc.to_csv(f'{OUT}/correspondence_corrected_n{len(dfc)}.csv', index=False)

print(f"\n=== correspondence by kind ===")
for k in ['warp0','warp2','warp4','ovl0.75','ovl0.25','same-cat','cross']:
    s = dfc[dfc.kind==k]
    if len(s): print(f"  {k:10} n={len(s):3d}  n_match {s.n_match.median():7.1f}  inlier {s.inlier.median():.3f}")

print(f"\n=== two regimes ===")
for rg in ['same-image','different-image']:
    s = dfc[dfc.regime==rg]
    print(f"\n  --- {rg} (n={len(s)}, |A-B| {s.endpoint_dist.median():.1f}) ---")
    for adv,lab in [('adv_sym_pixel','OT-sym − pixel'),('adv_sym_tex','OT-sym − texton'),
                    ('adv_ot_pixel','OT − pixel')]:
        w=int((s[adv]>0).sum()); p=wilcoxon(s[adv], alternative='greater').pvalue
        print(f"    {lab:16} median {s[adv].median():+.4f}  {w}/{len(s)}  p={p:.2e}")

s = dfc[dfc.regime=='different-image']
q = pd.qcut(s.endpoint_dist, 3, labels=['near','mid','far'])
print(f"\n=== control: advantage vs endpoint distance (different-image only) ===")
for lvl in q.cat.categories:
    t = s[q==lvl]
    print(f"  {lvl:6} n={len(t):3d}  |A-B| {t.endpoint_dist.median():5.1f}  "
          f"adv {t.adv_sym_pixel.median():+.4f}")

In [ ]:
def strong_warp(pil, lvl, seed):
    """Aggressive but correspondence-preserving distortions of one image.
    Pushes |A-B| high while SIFT+RANSAC can still recover a homography."""
    r = random.Random(seed); w,h = pil.size
    out = pil
    cfg = [
        dict(shift=40, ang=20, sc=1.35, persp=0.06, jit=0.15),
        dict(shift=60, ang=35, sc=1.6,  persp=0.10, jit=0.30),
        dict(shift=80, ang=55, sc=1.9,  persp=0.14, jit=0.45),
    ][lvl]
    out = out.transform(out.size, Image.AFFINE,
                        (1,0,r.randint(-cfg['shift'],cfg['shift']),
                         0,1,r.randint(-cfg['shift'],cfg['shift'])), Image.BILINEAR)
    out = out.rotate(r.uniform(-cfg['ang'],cfg['ang']), resample=Image.BILINEAR)
    # perspective, frame-filling
    d0 = cfg['persp']*w
    src=[(0,0),(w,0),(w,h),(0,h)]; dst=[(d0,d0),(w-d0,d0),(w,h),(0,h)]
    A_=[];B_=[]
    for (x,y),(X,Y) in zip(dst,src):
        A_.append([x,y,1,0,0,0,-X*x,-X*y]); B_.append(X)
        A_.append([0,0,0,x,y,1,-Y*x,-Y*y]); B_.append(Y)
    co = np.linalg.solve(np.array(A_), np.array(B_))
    out = out.transform(out.size, Image.PERSPECTIVE, co, Image.BILINEAR)
    # zoom to remove any border, then photometric jitter (raises |A-B| further)
    sc = cfg['sc']; cw,ch = int(w/sc), int(h/sc); l,t = (w-cw)//2,(h-ch)//2
    out = out.crop((l,t,l+cw,t+ch)).resize((w,h), Image.BILINEAR)
    a = np.asarray(out).astype(float)/255.
    j = cfg['jit']
    a = np.clip((a - 0.5)*(1+r.uniform(-j,j)) + 0.5 + r.uniform(-j,j)*0.5, 0, 1)
    return Image.fromarray((a*255).astype(np.uint8))

random.seed(21)
extra=[]
for lvl in range(3):
    for i in range(30):
        extra.append(dict(kind=f'strong{lvl}', pa=dtd_path(random.choice(cats), 77000+lvl*100+i),
                          lvl=lvl, seed=77000+lvl*100+i))

rows2=[]; t0=time.time()
for k,rec in enumerate(extra):
    A = Image.open(rec['pa']).convert('RGB').resize((IMG_SIZE,)*2)
    B = strong_warp(A, rec['lvl'], rec['seed'])
    ir, nm = corr_matches(A,B)
    d = build_d(cnt_load(A), cnt_load(B))
    mids = {m: to_pil(fn(d,0.5)) for m,fn in FNS.items()}
    R = {m: realism_d(f) for m,f in zip(METHODS, dino_feats([mids[m] for m in METHODS]))}
    fA,fB = dino_feats([A,B])
    rows2.append(dict(pair=1000+k, kind=rec['kind'], regime='same-image-strong',
                      inlier=ir, n_match=nm, endpoint_dist=float(np.linalg.norm(fA-fB)),
                      **{f'R_{m}': R[m] for m in METHODS},
                      adv_sym_pixel=R['ot_sym']-R['pixel_linear'],
                      adv_sym_tex=R['ot_sym']-R['texton_linear'],
                      adv_ot_pixel=R['ot']-R['pixel_linear']))
    if k%20==0: print(f"  {k}/{len(extra)} ({(time.time()-t0)/60:.1f}m)")

dfs2 = pd.DataFrame(rows2)
dfall = pd.concat([dfc, dfs2], ignore_index=True)
dfall.to_csv(f'{OUT}/correspondence_with_strong_n{len(dfall)}.csv', index=False)

print(f"\n=== did the strong warps achieve high |A-B| WITH correspondence? ===")
print(f"  {'kind':18} {'n':>4} {'n_match':>9} {'inlier':>8} {'|A-B|':>8}")
for k in ['strong0','strong1','strong2']:
    s = dfs2[dfs2.kind==k]
    print(f"  {k:18} {len(s):4d} {s.n_match.median():9.1f} {s.inlier.median():8.3f} "
          f"{s.endpoint_dist.median():8.1f}")
for rg,lbl in [('same-image','same-image (mild)'), ('different-image','different-image')]:
    s = dfc[dfc.regime==rg]
    print(f"  {lbl:18} {len(s):4d} {s.n_match.median():9.1f} {s.inlier.median():8.3f} "
          f"{s.endpoint_dist.median():8.1f}")

print(f"\n=== the decisive test: matched |A-B|, opposite correspondence ===")
lo, hi = dfs2.endpoint_dist.quantile([0.25,0.75])
band_corr = dfs2[(dfs2.endpoint_dist>=lo)&(dfs2.endpoint_dist<=hi)]
band_none = dfc[(dfc.regime=='different-image') &
                (dfc.endpoint_dist>=lo)&(dfc.endpoint_dist<=hi)]
print(f"  band |A-B| in [{lo:.1f}, {hi:.1f}]")
for lbl, s in [('correspondence YES', band_corr), ('correspondence NO ', band_none)]:
    if len(s)<5: print(f"  {lbl}: only n={len(s)}, too few"); continue
    w=int((s.adv_sym_pixel>0).sum()); p=wilcoxon(s.adv_sym_pixel, alternative='greater').pvalue
    print(f"  {lbl}  n={len(s):3d}  |A-B| {s.endpoint_dist.median():5.1f}  "
          f"n_match {s.n_match.median():6.1f}  adv {s.adv_sym_pixel.median():+.4f}  "
          f"{w}/{len(s)} p={p:.2e}")

print(f"\n=== advantage vs endpoint distance, WITHIN the correspondence-present set ===")
from scipy.stats import spearmanr
q = pd.qcut(dfs2.endpoint_dist, 3, labels=['near','mid','far'])
for lvl in q.cat.categories:
    t = dfs2[q==lvl]
    print(f"  {lvl:6} n={len(t):3d}  |A-B| {t.endpoint_dist.median():5.1f}  adv {t.adv_sym_pixel.median():+.4f}")
print(f"  spearman(adv, |A-B|) within correspondence-present: "
      f"{spearmanr(dfs2.endpoint_dist, dfs2.adv_sym_pixel).correlation:+.3f}")

In [ ]:
CROP = IMG_SIZE
def grid_crops(path, crop=CROP):
    pil = Image.open(path).convert('RGB'); W,H = pil.size
    return [pil.crop((x,y,x+crop,y+crop))
            for y in range(0,H-crop+1,crop) for x in range(0,W-crop+1,crop)]

usable=[]
for c in cats:
    for p in sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')):
        W,H = Image.open(p).size
        n = (W//CROP)*(H//CROP)
        if n >= 4: usable.append((c,p,n))
print(f"{len(usable)} images with >=4 disjoint {CROP}px crops")

N_PAIRS, CAT_BANK = 150, 40
rng = np.random.default_rng(0)
sel = [usable[i] for i in rng.choice(len(usable), size=N_PAIRS, replace=False)]
used = {p for _,p,_ in sel}
by_cat = {}
for c,p,n in usable: by_cat.setdefault(c,[]).append(p)

cat_bank = {}
for c in sorted({s[0] for s in sel}):
    pool = [p for p in by_cat[c] if p not in used]
    rng.shuffle(pool); crops=[]
    for p in pool:
        crops += grid_crops(p)
        if len(crops) >= CAT_BANK: break
    cat_bank[c] = dino_feats(crops[:CAT_BANK]) if crops else None
print(f"{sum(v is not None for v in cat_bank.values())} category banks")

def bank_dist(f, bank, k=5):
    bn = bank/(np.linalg.norm(bank,axis=1,keepdims=True)+1e-9)
    fn = f/(np.linalg.norm(f)+1e-9)
    d = 1.0 - bn@fn
    return float(np.sort(d)[:min(k,len(d))].mean())

rows=[]; t0=time.time()
for i,(cat,path,ncrop) in enumerate(sel):
    crops = grid_crops(path)
    idx = rng.permutation(len(crops))
    imA, imB = crops[idx[0]], crops[idx[1]]
    img_bank = dino_feats([crops[j] for j in idx[2:4]])
    cb = cat_bank.get(cat)
    if cb is None: continue

    tA = cnt_load(imA)
    fA, fB = dino_feats([imA]), dino_feats([imB])
    rt = to_pil(model.decode(model.splat(model.encode(tA))))
    fRT = dino_feats([rt])[0]

    d = build_d(tA, cnt_load(imB))
    rec = dict(pair=i, cat=cat,
               real_img=bank_dist(fA[0], img_bank, k=2), real_cat=bank_dist(fA[0], cb),
               floor_img=bank_dist(fRT, img_bank, k=2), floor_cat=bank_dist(fRT, cb),
               c_src=contrast(imA), c_floor=contrast(rt))
    mids = {m: to_pil(fn(d,0.5)) for m,fn in FNS.items()}
    fM = dino_feats([mids[m] for m in METHODS])
    for m,f in zip(METHODS, fM):
        rec[f'{m}_img'] = bank_dist(f, img_bank, k=2)
        rec[f'{m}_cat'] = bank_dist(f, cb)
        dA = 1.0-float(f@fA[0]/((np.linalg.norm(f)*np.linalg.norm(fA[0]))+1e-9))
        dB = 1.0-float(f@fB[0]/((np.linalg.norm(f)*np.linalg.norm(fB[0]))+1e-9))
        rec[f'{m}_balance']  = abs(dA-dB)/(dA+dB+1e-9)
        rec[f'{m}_distinct'] = min(dA,dB)
        rec[f'{m}_contrast'] = contrast(mids[m])
    rows.append(rec)
    if i%25==0: print(f"  {i}/{len(sel)} ({(time.time()-t0)/60:.1f}m)")

dfs = pd.DataFrame(rows); dfs.to_csv(f'{OUT}/selfsup_corrected_n{len(dfs)}.csv', index=False)

print(f"\n=== SAME-TEXTURE SELF-SUPERVISION, corrected (n={len(dfs)}) ===")
print(f"  real held-out crop   img-bank {dfs.real_img.median():.4f}  cat-bank {dfs.real_cat.median():.4f}")
print(f"  round-trip floor     img-bank {dfs.floor_img.median():.4f}  cat-bank {dfs.floor_cat.median():.4f}"
      f"   ({dfs.floor_img.median()/dfs.real_img.median():.2f}x real)")
print(f"  contrast: source {dfs.c_src.median():.1f} -> round-trip {dfs.c_floor.median():.1f}")
print(f"\n  {'method':16} {'in-family↓':>11} {'cat-bank↓':>10} {'balance↓':>9} {'distinct↑':>10} {'contrast':>9}")
for m in METHODS:
    print(f"  {m:16} {dfs[f'{m}_img'].median():11.4f} {dfs[f'{m}_cat'].median():10.4f} "
          f"{dfs[f'{m}_balance'].median():9.4f} {dfs[f'{m}_distinct'].median():10.4f} "
          f"{dfs[f'{m}_contrast'].median():9.1f}")

print(f"\n=== ot_sym vs each ===")
for base in ['ot','pixel_linear','texton_linear']:
    line = f"  vs {base:14}"
    for key,alt in [('img','less'),('balance','less'),('distinct','greater')]:
        a,b = dfs[f'ot_sym_{key}'], dfs[f'{base}_{key}']
        w=int(((a<b) if alt=='less' else (a>b)).sum())
        p=wilcoxon(a,b,alternative=alt).pvalue
        line += f"  {key:9}{w:3d}/{len(dfs)} p={p:.1e}"
    print(line)

In [ ]:
N_AX = 80
rng_ax = np.random.default_rng(2)

PAIRS_AX = []
for i in range(N_AX):
    a,b = random.Random(9000+i).sample(cats,2)
    PAIRS_AX.append((dtd_path(a,9000+i), dtd_path(b,12000+i)))

def dino_dist(p1, p2):
    f = dino_feats([p1,p2])
    return float(np.linalg.norm(f[0]-f[1]))

rows=[]; t0=time.time()
for i,(pa,pb) in enumerate(PAIRS_AX):
    pilA = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    pilB = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    tA, tB = cnt_load(pilA), cnt_load(pilB)
    d_ab, d_ba, d_aa = build_d(tA,tB), build_d(tB,tA), build_d(tA,tA)
    # scale reference: how far apart are the two sources?
    rec = dict(pair=i, ab_dist=dino_dist(pilA, pilB))
    for m,fn in FNS.items():
        rec[f'{m}_identity'] = dino_dist(to_pil(fn(d_aa,0.5)), pilA)
        rec[f'{m}_ep0']      = dino_dist(to_pil(fn(d_ab,0.0)), pilA)
        rec[f'{m}_ep1']      = dino_dist(to_pil(fn(d_ab,1.0)), pilB)
        s1 = to_pil(fn(d_ab,0.5)); s2 = to_pil(fn(d_ba,0.5))
        rec[f'{m}_symmetry'] = dino_dist(s1, s2)
        q1,q3 = to_pil(fn(d_ab,0.25)), to_pil(fn(d_ab,0.75))
        d_q = build_d(cnt_load(q1), cnt_load(q3))
        rec[f'{m}_subdiv']   = dino_dist(to_pil(fn(d_q,0.5)), s1)
        rec[f'{m}_seed']     = dino_dist(to_pil(fn(d_ab,0.5)), to_pil(fn(d_ab,0.5)))
    rows.append(rec)
    if i%20==0: print(f"  {i}/{N_AX} ({(time.time()-t0)/60:.1f}m)")

dfa = pd.DataFrame(rows); dfa.to_csv(f'{OUT}/consistency_corrected_n{N_AX}.csv', index=False)

TESTS = [('identity','M(A,A,.5) vs A'), ('ep0','M(A,B,0) vs A'), ('ep1','M(A,B,1) vs B'),
         ('symmetry','M(A,B,.5) vs M(B,A,.5)'), ('subdiv','direct vs composed'),
         ('seed','re-run stability')]

print(f"\n=== AXIOMATIC CONSISTENCY, corrected (n={N_AX}) ===")
print(f"  DINOv2 distance; for scale, |A-B| = {dfa.ab_dist.median():.2f}\n")
print(f"  {'test':26} " + "".join(f"{m[:13]:>15}" for m in METHODS))
for key,lab in TESTS:
    print(f"  {lab:26} " + "".join(f"{dfa[f'{m}_{key}'].median():15.3f}" for m in METHODS))

print(f"\n  as a fraction of |A-B| (lower = more consistent)")
print(f"  {'test':26} " + "".join(f"{m[:13]:>15}" for m in METHODS))
for key,lab in TESTS:
    print(f"  {lab:26} " + "".join(f"{(dfa[f'{m}_{key}']/dfa.ab_dist).median():15.3f}" for m in METHODS))

print(f"\n=== ot_sym vs each (lower better) ===")
for base in ['ot','pixel_linear','texton_linear']:
    line = f"  vs {base:14}"
    for key in ['symmetry','subdiv','ep1']:
        a,b = dfa[f'ot_sym_{key}'], dfa[f'{base}_{key}']
        w=int((a<b).sum()); p=wilcoxon(a,b,alternative='less').pvalue
        line += f"  {key:9}{w:3d}/{N_AX} p={p:.1e}"
    print(line)

In [ ]:
def zoom_center(pil, f=1.5):
    w,h = pil.size; cw,ch = int(w/f), int(h/f); l,t = (w-cw)//2,(h-ch)//2
    return pil.crop((l,t,l+cw,t+ch)).resize((w,h), Image.BILINEAR)
def perspective_warp(pil, strength=0.10, pre=1.25, post=1.15):
    pil = zoom_center(pil, pre); w,h = pil.size; dx,dy = strength*w, strength*h
    src=[(0,0),(w,0),(w,h),(0,h)]; dst=[(dx,dy),(w-dx,dy),(w,h),(0,h)]
    A_=[];B_=[]
    for (x,y),(X,Y) in zip(dst,src):
        A_.append([x,y,1,0,0,0,-X*x,-X*y]); B_.append(X)
        A_.append([0,0,0,x,y,1,-Y*x,-Y*y]); B_.append(Y)
    co = np.linalg.solve(np.array(A_), np.array(B_))
    return zoom_center(pil.transform((w,h), Image.PERSPECTIVE, co, Image.BILINEAR), post)
def to_gray3(pil):
    g = pil.convert('L'); return Image.merge('RGB',(g,g,g))
def chroma_only(pil):
    y,cb,cr = pil.convert('YCbCr').split()
    return Image.merge('YCbCr',(Image.new('L',pil.size,128),cb,cr)).convert('RGB')
def hist_match_gray(pil):
    g = np.asarray(pil.convert('L')).astype(float).ravel()
    out = np.empty_like(g); out[g.argsort()] = np.linspace(0,255,g.size)
    a = out.reshape(pil.size[1], pil.size[0]).astype(np.uint8)
    return Image.merge('RGB',(Image.fromarray(a),)*3)
def phase_only(pil):
    g = np.asarray(pil.convert('L')).astype(float)
    amp = np.abs(np.fft.fft2(np.random.default_rng(0).standard_normal(g.shape)))
    rec = np.real(np.fft.ifft2(amp*np.exp(1j*np.angle(np.fft.fft2(g)))))
    rec = (rec-rec.min())/(np.ptp(rec)+1e-9)*255
    return Image.merge('RGB',(Image.fromarray(rec.astype(np.uint8)),)*3)
def spectrum_only(pil):
    g = np.asarray(pil.convert('L')).astype(float)
    ph = np.angle(np.fft.fft2(np.random.default_rng(1).standard_normal(g.shape)))
    rec = np.real(np.fft.ifft2(np.abs(np.fft.fft2(g))*np.exp(1j*ph)))
    rec = (rec-rec.min())/(np.ptp(rec)+1e-9)*255
    return Image.merge('RGB',(Image.fromarray(rec.astype(np.uint8)),)*3)

PLIST = PAIRS[:100]          # from the corrected realism cell

def run_condition(tf, tag):
    imgs=[]
    for i in range(300):
        pil = Image.open(dtd_path(random.choice(cats), 50000+i)).convert('RGB').resize((IMG_SIZE,)*2)
        imgs.append(pil if tf is None else tf(pil))
    Phi = dino_feats(imgs); Phi = Phi[~np.isnan(Phi).any(axis=1)]
    D = np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
    rad = np.sort(D,axis=1)[:,2]; keep = rad <= np.median(rad)
    Pr, rr = Phi[keep], rad[keep]
    def rl(f):
        dd=np.linalg.norm(Pr-f,axis=1)+1e-9; return float(np.max(rr/dd))
    mids={m:[] for m in METHODS}; t0=time.time()
    for k,(pa,pb) in enumerate(PLIST):
        A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
        B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
        if tf is not None: A,B = tf(A), tf(B)
        d = build_d(cnt_load(A), cnt_load(B))
        for m,fn in FNS.items(): mids[m].append(to_pil(fn(d,0.5)))
        if k%40==0: print(f"  [{tag}] {k}/{len(PLIST)} ({(time.time()-t0)/60:.1f}m)")
    return {m: np.array([rl(f) for f in dino_feats(mids[m])]) for m in METHODS}

CONDS = [(None,'full'), (zoom_center,'zoom'), (perspective_warp,'perspective'),
         (to_gray3,'luminance'), (chroma_only,'chroma'), (hist_match_gray,'hist-matched'),
         (phase_only,'phase-only'), (spectrum_only,'spectrum-only')]
resC = {}
for tf,tag in CONDS:
    print(f"=== {tag} ===")
    resC[tag] = run_condition(tf, tag)
np.save(f'{OUT}/conditions_corrected.npy', resC, allow_pickle=True)

print(f"\n=== REALISM ACROSS CONDITIONS (n={len(PLIST)}, DINOv2) ===")
print(f"  {'condition':14} " + "".join(f"{m[:13]:>15}" for m in METHODS))
for _,tag in CONDS:
    print(f"  {tag:14} " + "".join(f"{resC[tag][m].mean():15.3f}" for m in METHODS))

print(f"\n  {'condition':14} {'%R≥1 ot_sym':>13} {'vs pixel':>20} {'vs ot':>20}")
for _,tag in CONDS:
    sc = resC[tag]; a,b,c = sc['ot_sym'], sc['pixel_linear'], sc['ot']
    w1=int((a>b).sum()); p1=wilcoxon(a,b,alternative='greater').pvalue
    w2=int((a>c).sum()); p2=wilcoxon(a,c,alternative='greater').pvalue
    print(f"  {tag:14} {100*(a>=1).mean():12.0f}% {w1:8d}/{len(a)} p={p1:.1e} "
          f"{w2:8d}/{len(a)} p={p2:.1e}")

In [ ]:
N_SUB = 100
PLIST_SUB = PAIRS[:N_SUB]

rows=[]; t0=time.time()
for k,(pa,pb) in enumerate(PLIST_SUB):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d = build_d(cnt_load(A), cnt_load(B))
    rec = dict(pair=k, ab_dist=dino_dist(A,B))
    for m,fn in FNS.items():
        direct = to_pil(fn(d,0.5))

        # one level: midpoint of the quarter and three-quarter points
        q1, q3 = to_pil(fn(d,0.25)), to_pil(fn(d,0.75))
        comp1 = to_pil(fn(build_d(cnt_load(q1), cnt_load(q3)), 0.5))
        rec[f'{m}_div1'] = dino_dist(comp1, direct)

        # two levels
        e1,e2 = to_pil(fn(d,0.125)), to_pil(fn(d,0.375))
        e3,e4 = to_pil(fn(d,0.625)), to_pil(fn(d,0.875))
        m1 = to_pil(fn(build_d(cnt_load(e1), cnt_load(e2)), 0.5))
        m2 = to_pil(fn(build_d(cnt_load(e3), cnt_load(e4)), 0.5))
        comp2 = to_pil(fn(build_d(cnt_load(m1), cnt_load(m2)), 0.5))
        rec[f'{m}_div2'] = dino_dist(comp2, direct)

        # cycle: A->B at 0.5 vs B->A at 0.5, then re-encode and compare
        rev = to_pil(fn(build_d(cnt_load(B), cnt_load(A)), 0.5))
        rec[f'{m}_cycle'] = dino_dist(rev, direct)

        # baseline for scale: a re-run of the same call
        rec[f'{m}_seed'] = dino_dist(to_pil(fn(d,0.5)), direct)
    rows.append(rec)
    if k%20==0: print(f"  {k}/{N_SUB} ({(time.time()-t0)/60:.1f}m)")

dfe = pd.DataFrame(rows); dfe.to_csv(f'{OUT}/subdivision_corrected_n{N_SUB}.csv', index=False)

print(f"\n=== SUBDIVISION / COMPOSITION, corrected (n={N_SUB}) ===")
print(f"  DINOv2 distance to the direct midpoint; |A-B| = {dfe.ab_dist.median():.2f}\n")
print(f"  {'test':22} " + "".join(f"{m[:13]:>15}" for m in METHODS))
for key,lab in [('seed','re-run (noise floor)'), ('cycle','reversed order'),
                ('div1','1 level composed'), ('div2','2 levels composed')]:
    print(f"  {lab:22} " + "".join(f"{dfe[f'{m}_{key}'].median():15.3f}" for m in METHODS))

print(f"\n  as a fraction of |A-B|")
print(f"  {'test':22} " + "".join(f"{m[:13]:>15}" for m in METHODS))
for key,lab in [('seed','re-run (noise floor)'), ('cycle','reversed order'),
                ('div1','1 level composed'), ('div2','2 levels composed')]:
    print(f"  {lab:22} " + "".join(f"{(dfe[f'{m}_{key}']/dfe.ab_dist).median():15.3f}" for m in METHODS))

print(f"\n=== composition error above each method's own seed noise ===")
print(f"  {'method':16} {'div1 - seed':>13} {'div2 - seed':>13} {'div2/div1':>11}")
for m in METHODS:
    d1 = (dfe[f'{m}_div1']-dfe[f'{m}_seed']).median()
    d2 = (dfe[f'{m}_div2']-dfe[f'{m}_seed']).median()
    print(f"  {m:16} {d1:13.3f} {d2:13.3f} {(dfe[f'{m}_div2']/dfe[f'{m}_div1']).median():11.3f}")

print(f"\n=== ot_sym vs each (lower better) ===")
for base in ['ot','pixel_linear','texton_linear']:
    line = f"  vs {base:14}"
    for key in ['div1','div2','cycle']:
        a,b = dfe[f'ot_sym_{key}'], dfe[f'{base}_{key}']
        w=int((a<b).sum()); p=wilcoxon(a,b,alternative='less').pvalue
        line += f"  {key:6}{w:3d}/{N_SUB} p={p:.1e}"
    print(line)

In [ ]:
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
ETAS = [0.0, 0.25, 0.5, 0.75, 1.0]

def path_row(fn, d, A, B):
    """Full path with DECODED endpoints, so every panel goes through the same pipeline."""
    dA = build_d(cnt_load(A), cnt_load(A)); dB = build_d(cnt_load(B), cnt_load(B))
    out = [to_pil(fn(dA, 0.0))]
    out += [to_pil(fn(d, e)) for e in ETAS[1:-1]]
    out += [to_pil(fn(dB, 0.0))]
    return out

# --- Figure 1: paths, OT above OT-sym, several pairs ---
SHOW = [(dtd_path('dotted',700+i), dtd_path('honeycombed',800+i)) for i in range(2)] + \
       [(dtd_path('zigzagged',900+i), dtd_path('bubbly',950+i)) for i in range(2)]

fig, axes = plt.subplots(len(SHOW)*2, len(ETAS)+2, figsize=(2.9*(len(ETAS)+2), 2.9*len(SHOW)*2))
for i,(pa,pb) in enumerate(SHOW):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d = build_d(cnt_load(A), cnt_load(B))
    for j,(nm,fn) in enumerate([('OT',interp_ot), ('OT-sym',interp_ot_sym)]):
        row = i*2+j
        mids = path_row(fn, d, A, B)
        panels = [('A',A)] + [(f'η={e}',m) for e,m in zip(ETAS,mids)] + [('B',B)]
        for c,(lbl,im) in enumerate(panels):
            ax = axes[row,c]; ax.imshow(im); ax.axis('off')
            ax.set_title(f"{lbl}   {contrast(im):.0f}", fontsize=8)
        axes[row,0].text(-0.14,0.5, f"pair {i+1}\n{nm}", transform=axes[row,0].transAxes,
                         rotation=90, va='center', ha='center', fontsize=10, weight='bold')
plt.suptitle("OT vs OT-sym along the path (endpoints decoded; contrast under each panel)",
             fontsize=13, y=1.003)
plt.tight_layout(); plt.savefig(f'{FIG}/ot_vs_otsym_paths.png', dpi=150, bbox_inches='tight'); plt.show()

# --- Figure 2: the order-dependence, which is the actual point ---
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for i,(pa,pb) in enumerate(SHOW[:3]):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d_ab = build_d(cnt_load(A), cnt_load(B))
    d_ba = build_d(cnt_load(B), cnt_load(A))
    ot_ab, ot_ba   = to_pil(interp_ot(d_ab,0.5)),     to_pil(interp_ot(d_ba,0.5))
    sym_ab, sym_ba = to_pil(interp_ot_sym(d_ab,0.5)), to_pil(interp_ot_sym(d_ba,0.5))
    panels = [('A',A), ('OT  M(A,B)',ot_ab), ('OT  M(B,A)',ot_ba),
              ('OT-sym  M(A,B)',sym_ab), ('OT-sym  M(B,A)',sym_ba)]
    for c,(lbl,im) in enumerate(panels):
        axes[i,c].imshow(im); axes[i,c].axis('off'); axes[i,c].set_title(lbl, fontsize=10)
    axes[i,0].set_title(f"A   (B below)", fontsize=10)
plt.suptitle("Order dependence: OT's midpoint changes when the arguments swap; OT-sym's does not",
             fontsize=13)
plt.tight_layout(); plt.savefig(f'{FIG}/ot_order_dependence.png', dpi=150, bbox_inches='tight'); plt.show()

# --- Figure 3: quantified, on the pairs already measured ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sym = dfe[[f'{m}_cycle' for m in METHODS]].div(dfe.ab_dist, axis=0)
sym.columns = METHODS
bp = ax[0].boxplot([sym[m] for m in METHODS], labels=METHODS, patch_artist=True, showfliers=False)
for patch,c in zip(bp['boxes'], ['#4C72B0','#DD8452','#55A868','#C44E52']):
    patch.set_facecolor(c); patch.set_alpha(0.8)
ax[0].set_ylabel("|M(A,B) − M(B,A)| / |A−B|")
ax[0].set_title("Order dependence (lower = symmetric)", fontsize=11)
ax[0].tick_params(axis='x', rotation=20)

orient = [dfo[f'{m}_th_err'] for m in METHODS]
bp2 = ax[1].boxplot(orient, labels=METHODS, patch_artist=True, showfliers=False)
for patch,c in zip(bp2['boxes'], ['#4C72B0','#DD8452','#55A868','#C44E52']):
    patch.set_facecolor(c); patch.set_alpha(0.8)
ax[1].set_ylabel("midpoint orientation error (°)")
ax[1].set_title("Parametric benchmark: known midpoint", fontsize=11)
ax[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(f'{FIG}/otsym_summary.png', dpi=150, bbox_inches='tight'); plt.show()
print("saved to", FIG)

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(19, 10))
for i,(pa,pb) in enumerate(SHOW[:3]):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d_ab = build_d(cnt_load(A), cnt_load(B)); d_ba = build_d(cnt_load(B), cnt_load(A))
    panels = [('A',A),
              ('OT  M(A,B)',   to_pil(interp_ot(d_ab,0.5))),
              ('OT  M(B,A)',   to_pil(interp_ot(d_ba,0.5))),
              ('OT-sym  M(A,B)', to_pil(interp_ot_sym(d_ab,0.5))),
              ('OT-sym  M(B,A)', to_pil(interp_ot_sym(d_ba,0.5))),
              ('B',B)]
    for c,(lbl,im) in enumerate(panels):
        axes[i,c].imshow(im); axes[i,c].axis('off'); axes[i,c].set_title(lbl, fontsize=10)
plt.suptitle("Swapping the arguments changes OT's midpoint but not OT-sym's", fontsize=13)
plt.tight_layout(); plt.savefig(f'{FIG}/ot_order_dependence_v2.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
!pip install -q lpips
import lpips as lpips_lib
lpips_fn = lpips_lib.LPIPS(net='alex').to(device).eval()
for p in lpips_fn.parameters(): p.requires_grad_(False)

@torch.no_grad()
def lp(p1, p2):
    """LPIPS between two PIL images, in [-1,1] as the net expects."""
    def prep(p):
        a = np.asarray(p.convert('RGB').resize((IMG_SIZE,)*2)).astype(np.float32)/255.
        return torch.from_numpy(a).permute(2,0,1).unsqueeze(0).to(device)*2-1
    return float(lpips_fn(prep(p1), prep(p2)).item())

N_LP = 200
ETAS_P = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
rng_lp = random.Random(31337)
PAIRS_LP = []
while len(PAIRS_LP) < N_LP:
    a,b = rng_lp.sample(cats,2)
    PAIRS_LP.append((dtd_path(a,31000+len(PAIRS_LP)), dtd_path(b,52000+len(PAIRS_LP))))

rows=[]; t0=time.time()
for k,(pa,pb) in enumerate(PAIRS_LP):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d  = build_d(cnt_load(A), cnt_load(B))
    dA = build_d(cnt_load(A), cnt_load(A)); dB = build_d(cnt_load(B), cnt_load(B))
    rec = dict(pair=k, d_AB=lp(A,B))
    for m,fn in FNS.items():
        # decoded endpoints, so the path is measured in one consistent space
        pathi = [to_pil(fn(dA,0.0))] + [to_pil(fn(d,e)) for e in ETAS_P[1:-1]] + [to_pil(fn(dB,0.0))]
        E0, E1 = pathi[0], pathi[-1]
        mid = pathi[ETAS_P.index(0.5)]
        dmA, dmB = lp(mid,E0), lp(mid,E1)
        rec[f'{m}_ratio']   = dmA/(dmA+dmB+1e-9)              # ideal 0.5
        rec[f'{m}_mid_err'] = abs(dmA/(dmA+dmB+1e-9) - 0.5)
        # path geometry against the endpoint separation
        dE = lp(E0,E1)+1e-9
        rec[f'{m}_detour']  = (dmA+dmB)/dE                    # 1.0 = straight
        steps = [lp(pathi[i], pathi[i+1]) for i in range(len(pathi)-1)]
        rec[f'{m}_mono']    = float(np.mean(np.diff(
                                 np.cumsum([0]+steps)/ (sum(steps)+1e-9)) > 0))
        rec[f'{m}_smooth']  = float(np.std(steps)/(np.mean(steps)+1e-9))   # lower = even pacing
        rec[f'{m}_pathlen'] = sum(steps)/dE
    rows.append(rec)
    if k%25==0: print(f"  {k}/{N_LP} ({(time.time()-t0)/60:.1f}m)")

dfl = pd.DataFrame(rows); dfl.to_csv(f'{OUT}/lpips_corrected_n{N_LP}.csv', index=False)

print(f"\n=== LPIPS MIDPOINT & PATH, corrected (n={N_LP}) ===")
print(f"  {'method':16} {'ratio→0.5':>11} {'|err|↓':>9} {'detour↓':>9} {'pathlen↓':>10} {'pacing↓':>9} {'mono':>7}")
for m in METHODS:
    print(f"  {m:16} {dfl[f'{m}_ratio'].median():11.3f} {dfl[f'{m}_mid_err'].median():9.3f} "
          f"{dfl[f'{m}_detour'].median():9.3f} {dfl[f'{m}_pathlen'].median():10.3f} "
          f"{dfl[f'{m}_smooth'].median():9.3f} {dfl[f'{m}_mono'].mean():7.2f}")

print(f"\n=== paired comparisons (lower better) ===")
for tgt in ['ot','ot_sym']:
    print(f"  --- {tgt} vs ---")
    for base in [m for m in METHODS if m!=tgt]:
        line = f"    {base:14}"
        for key in ['mid_err','detour','smooth']:
            a,b = dfl[f'{tgt}_{key}'], dfl[f'{base}_{key}']
            w=int((a<b).sum()); p=wilcoxon(a,b,alternative='less').pvalue
            line += f"  {key:8}{w:3d}/{N_LP} p={p:.1e}"
        print(line)

In [ ]:
from scipy import linalg
from sklearn.neighbors import NearestNeighbors

def fid(fr, fg):
    mu_r, mu_g = fr.mean(0), fg.mean(0)
    cr, cg = np.cov(fr, rowvar=False), np.cov(fg, rowvar=False)
    covmean, _ = linalg.sqrtm(cr.dot(cg), disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(((mu_r-mu_g)**2).sum() + np.trace(cr + cg - 2*covmean))

def _knn_radii(f, k=3):
    nn = NearestNeighbors(n_neighbors=k+1).fit(f)
    d,_ = nn.kneighbors(f)
    return d[:, k]

def precision_recall(fr, fg, k=3):
    rr, rg = _knn_radii(fr,k), _knn_radii(fg,k)
    d_gr = np.linalg.norm(fg[:,None,:]-fr[None,:,:], axis=2)   # gen x real
    precision = float((d_gr <= rr[None,:]).any(axis=1).mean())  # gen inside real manifold
    recall    = float((d_gr <= rg[:,None]).any(axis=0).mean())  # real inside gen manifold
    return precision, recall

def coverage(fr, fg, k=3):
    rr = _knn_radii(fr, k)
    d_rg = np.linalg.norm(fr[:,None,:]-fg[None,:,:], axis=2)    # real x gen
    return float((d_rg.min(axis=1) <= rr).mean())

def one_nna(fr, fg):
    X = np.vstack([fr, fg]); y = np.array([0]*len(fr) + [1]*len(fg))
    D = np.linalg.norm(X[:,None,:]-X[None,:,:], axis=2)
    np.fill_diagonal(D, np.inf)
    return float((y[D.argmin(1)] == y).mean())

print("metrics ready")

In [ ]:
N_PAIRS_FID = 500        # 1000 if you want to match the original exactly
SEED_FID = 0
rnd = random.Random(SEED_FID)
pairs_fid = []
while len(pairs_fid) < N_PAIRS_FID:
    ca, cb = rnd.sample(cats, 2)
    pairs_fid.append((dtd_path(ca, len(pairs_fid)), dtd_path(cb, len(pairs_fid)+9999)))

mids = {m: [] for m in METHODS}
real_imgs = []
t0 = time.time()
for k,(pa,pb) in enumerate(pairs_fid):
    try:
        d = build_d(cnt_load(pa), cnt_load(pb))
        for m,fn in FNS.items():
            mids[m].append(to_pil(fn(d, 0.5)))
        # reference = the SOURCE textures decoded through CNT, matching the original protocol
        real_imgs.append(to_pil(decode_interp(d["blob_a"], d["bf_a"], d["si_a"])))
        real_imgs.append(to_pil(decode_interp(d["blob_b"], d["bf_b"], d["si_b"])))
    except Exception as e:
        if k % 50 == 0: print(f"  skip {k}: {type(e).__name__}: {e}")
    if k % 50 == 0:
        el = time.time()-t0
        print(f"  {k}/{N_PAIRS_FID}  {el/60:.1f}m  eta {el/(k+1)*(N_PAIRS_FID-k-1)/60:.1f}m")

print("\ncounts:", {m: len(v) for m,v in mids.items()}, "| refs:", len(real_imgs))
real_feats = inc_feats(real_imgs)
print("real feats:", real_feats.shape)

rows = {}
for m in METHODS:
    fg = inc_feats(mids[m])
    n = min(len(real_feats), len(fg))
    fr_, fg_ = real_feats[:n], fg[:n]
    prec, rec = precision_recall(fr_, fg_)
    rows[m] = dict(FID=fid(fr_,fg_), NNA=one_nna(fr_,fg_), precision=prec,
                   recall=rec, coverage=coverage(fr_,fg_), n=n)

print(f"\n{'method':16}{'FID↓':>9}{'1-NNA→.5':>11}{'prec↑':>8}{'recall↑':>9}{'cov↑':>8}   (n={rows['ot']['n']})")
print("="*70)
for m in ['texton_linear','pixel_linear','ot','ot_sym']:
    x = rows[m]
    print(f"{m:16}{x['FID']:9.2f}{x['NNA']:11.3f}{x['precision']:8.3f}{x['recall']:9.3f}{x['coverage']:8.3f}")

print("\nbest per metric:")
for kk,f in {'FID':min,'precision':max,'recall':max,'coverage':max}.items():
    print(f"  {kk:10} -> {f(rows, key=lambda mm: rows[mm][kk])}")
print(f"  1-NNA(→0.5) -> {min(rows, key=lambda mm: abs(rows[mm]['NNA']-0.5))}")

print("\not_sym vs ot:")
for kk in ['FID','precision','recall','coverage']:
    a,b = rows['ot_sym'][kk], rows['ot'][kk]
    better = (a<b) if kk=='FID' else (a>b)
    print(f"  {kk:10} {b:8.3f} -> {a:8.3f}   {'better' if better else 'worse'}")

import json
json.dump({m:{kk:float(v) for kk,v in r.items()} for m,r in rows.items()},
          open(f'{OUT}/inception_with_sym_n{rows["ot"]["n"]}.json','w'), indent=2)